<a href="https://colab.research.google.com/github/papy-studio/Coding_week_Groupe-17/blob/Papy/LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.makedirs("data", exist_ok=True)

In [2]:
import sys
!{sys.executable} -m pip install lightgbm scikit-learn joblib pandas numpy

In [3]:
%pip install ucimlrepo
from ucimlrepo import fetch_ucirepo


Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Charger la data depuis le CSV
df = pd.read_csv('../notebooks/data/ObesityDataSet.csv')
df_original = df.copy()

# ── AVANT ────────────────────────────────────────────────────────────────────
print("=" * 50)
print("AVANT — Data originale")
print("=" * 50)
print(df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].head(3))
print(f"\nTypes : \n{df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].dtypes}")

# ── ENCODAGE ─────────────────────────────────────────────────────────────────
binary_cols = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

le = LabelEncoder()
for col in ['Gender', 'CAEC', 'CALC', 'MTRANS']:
    df[col] = le.fit_transform(df[col])

le_target = LabelEncoder()
df['NObeyesdad'] = le_target.fit_transform(df['NObeyesdad'])

# ── APRÈS ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("APRÈS — Data encodée")
print("=" * 50)
print(df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].head(3))
print(f"\nTypes : \n{df[['Gender', 'SMOKE', 'CAEC', 'NObeyesdad']].dtypes}")

# ── MÉMOIRE ───────────────────────────────────────────────────────────────────
def optimize_memory(df):
    before = df.memory_usage(deep=True).sum() / 1024
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    after = df.memory_usage(deep=True).sum() / 1024
    print(f"\n{'='*50}")
    print("OPTIMISATION MÉMOIRE")
    print(f"{'='*50}")
    print(f"Avant  : {before:.1f} KB")
    print(f"Après  : {after:.1f} KB")
    print(f"Gagné  : {((before-after)/before*100):.1f}%")
    return df

df = optimize_memory(df)

# ── SPLIT ─────────────────────────────────────────────────────────────────────
X = df.drop(columns=['NObeyesdad'])
y = df['NObeyesdad']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n{'='*50}")
print("SÉPARATION TRAIN / TEST")
print(f"{'='*50}")
print(f"Total lignes    : {len(df)}")
print(f"Train (80%)     : {X_train.shape[0]} lignes")
print(f"Test  (20%)     : {X_test.shape[0]} lignes")
print(f"\n✅ Data prête pour le ML !")

AVANT — Data originale
   Gender SMOKE       CAEC     NObeyesdad
0  Female    no  Sometimes  Normal_Weight
1  Female   yes  Sometimes  Normal_Weight
2    Male    no  Sometimes  Normal_Weight

Types : 
Gender        object
SMOKE         object
CAEC          object
NObeyesdad    object
dtype: object

APRÈS — Data encodée
   Gender  SMOKE  CAEC  NObeyesdad
0       0      0     2           1
1       0      1     2           1
2       1      0     2           1

Types : 
Gender        int64
SMOKE         int64
CAEC          int64
NObeyesdad    int64
dtype: object

OPTIMISATION MÉMOIRE
Avant  : 280.5 KB
Après  : 140.3 KB
Gagné  : 50.0%

SÉPARATION TRAIN / TEST
Total lignes    : 2111
Train (80%)     : 1688 lignes
Test  (20%)     : 423 lignes

✅ Data prête pour le ML !


In [6]:
X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

print("Data splits saved to 'data/' directory:")
print('data/X_train.csv')
print('data/X_test.csv')
print('data/y_train.csv')
print('data/y_test.csv')

Data splits saved to 'data/' directory:
data/X_train.csv
data/X_test.csv
data/y_train.csv
data/y_test.csv


In [7]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")

y_train = pd.read_csv("data/y_train.csv")
y_test = pd.read_csv("data/y_test.csv")

model = LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6)

model.fit(X_train, y_train.values.ravel())

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")
roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovr")

print("Accuracy :", accuracy)
print("Precision :", precision)
print("Recall :", recall)
print("F1-score :", f1)
print("ROC-AUC :", roc_auc)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000620 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2063
[LightGBM] [Info] Number of data points in the train set: 1688, number of used features: 16
[LightGBM] [Info] Start training from score -2.056021
[LightGBM] [Info] Start training from score -2.015199
[LightGBM] [Info] Start training from score -1.821828
[LightGBM] [Info] Start training from score -1.954836
[LightGBM] [Info] Start training from score -1.866779
[LightGBM] [Info] Start training from score -1.975979
[LightGBM] [Info] Start training from score -1.950661
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

In [8]:
import joblib
from pathlib import Path
Path("models").mkdir(exist_ok=True)
joblib.dump(model, "models/model.pkl")
print("✅ Sauvegardé !")

✅ Sauvegardé !
